# LSA Retriever Tests
Run LSA retrieval on the Cranfield-style dataset and inspect a few sample queries.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
import nltk

for resource in ("punkt", "stopwords", "wordnet"):
    nltk.download(resource, quiet=True)

def resolve_project_dir():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd / "traditional-retrieval"]
    for path in candidates:
        if (path / "data").exists() and (path / "source").exists():
            return path
    raise FileNotFoundError("Could not locate the traditional-retrieval project directory.")

project_dir = resolve_project_dir()
data_dir = project_dir / "data"
utils_dir = project_dir / "source" / "utils"

for path in (str(project_dir), str(utils_dir)):
    if path not in sys.path:
        sys.path.append(path)

from source.utils.textPreprocessing import TextPreprocessing
from source.utils.inverted_index import InvertedIndex
from source.utils.LSA import LSA

documents_path = data_dir / "documents.csv"
queries_path = data_dir / "queries.csv"

documents = pd.read_csv(documents_path)
queries = pd.read_csv(queries_path)

tp = TextPreprocessing()
processed = tp.process_dataframe(documents)
vocab = tp.get_vocab()

index = InvertedIndex(processed, vocab)
index._build()

lsa_dims = 200
lsa = LSA(processed, index, vocab, dims=lsa_dims)

In [2]:
# Basic checks on real dataset
assert len(documents) > 0
assert len(queries) > 0

# Run a few sample queries
sample_queries = queries.head(5).to_dict(orient="records")
doc_map = documents.set_index("id")["content"].to_dict()

results = []
for row in sample_queries:
    qid = row["id"]
    query_text = row["content"]
    hits = lsa.search(query_text, top_k=5)
    enriched_hits = [
        {
            "doc_id": int(doc_id),
            "score": float(score),
            "content": doc_map.get(doc_id, ""),
        }
        for doc_id, score in hits
    ]
    results.append({"qid": int(qid), "query": query_text, "top5": enriched_hits})

results_df = pd.DataFrame(results)
results_df

,qid,query,top5
0,1,what similarity laws must be obeyed when cons...,"[{'doc_id': 486, 'score': 0.6125456690788269, ..."
1,2,what are the structural and aeroelastic probl...,"[{'doc_id': 12, 'score': 0.8272419571876526, '..."
2,3,what problems of heat conduction in composite...,"[{'doc_id': 485, 'score': 0.7837008833885193, ..."
3,4,can a criterion be developed to show empirica...,"[{'doc_id': 236, 'score': 0.6481595039367676, ..."
4,5,what chemical kinetic system is applicable to...,"[{'doc_id': 367, 'score': 0.4208623170852661, ..."


# Evaluation Metrics (REL Ground Truth)
Compute MAP@10, NDCG@10, Precision/Recall/F1 at 10 using Cranfield REL files.

In [3]:
import math
import numpy as np

rel_dir = project_dir.parent / "cranfield-dataset" / "REL"
if rel_dir is None:
    raise FileNotFoundError("Could not locate the Cranfield REL directory.")

def load_relations(rel_folder):
    qrels = {}
    for name in os.listdir(rel_folder):
        if not name.endswith(".txt"):
            continue
        path = os.path.join(rel_folder, name)
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 3:
                    continue
                qid = int(parts[0])
                doc_id = int(parts[1])
                rel = float(parts[2])
                qrels.setdefault(qid, {})[doc_id] = rel
    return qrels

def normalize_relevance(rel_map):
    positive = [rel for rel in rel_map.values() if rel > 0]
    if not positive:
        return {}
    max_rel = max(positive)
    return {doc_id: (max_rel - rel + 1) if rel > 0 else 0.0 for doc_id, rel in rel_map.items()}

def precision_recall_f1_at_k(retrieved, relevant_set, k):
    if k <= 0:
        return 0.0, 0.0, 0.0
    top_k = retrieved[:k]
    if not top_k:
        return 0.0, 0.0, 0.0
    hit = sum(1 for doc_id in top_k if doc_id in relevant_set)
    precision = hit / k
    recall = hit / max(len(relevant_set), 1)
    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)
    return precision, recall, f1

def average_precision_at_k(retrieved, relevant_set, k):
    if not relevant_set:
        return 0.0
    ap_sum = 0.0
    hit = 0
    for i, doc_id in enumerate(retrieved[:k], start=1):
        if doc_id in relevant_set:
            hit += 1
            ap_sum += hit / i
    return ap_sum / len(relevant_set)

def ndcg_at_k(retrieved, rel_map, k):
    def dcg(scores):
        return sum((2 ** rel - 1) / math.log2(i + 2) for i, rel in enumerate(scores))
    gains = [rel_map.get(doc_id, 0.0) for doc_id in retrieved[:k]]
    ideal = sorted(rel_map.values(), reverse=True)[:k]
    dcg_val = dcg(gains)
    idcg_val = dcg(ideal)
    if idcg_val == 0:
        return 0.0
    return dcg_val / idcg_val

qrels = load_relations(rel_dir)

k = 10
precision_list = []
recall_list = []
f1_list = []
ap_list = []
ndcg_list = []

for row in queries.to_dict(orient="records"):
    qid = int(row["id"])
    query_text = row["content"]
    hits = lsa.search(query_text, top_k=k)
    retrieved = [int(doc_id) for doc_id, _ in hits]
    rel_map_raw = qrels.get(qid, {})
    rel_map = normalize_relevance(rel_map_raw)
    relevant_set = {doc_id for doc_id, rel in rel_map_raw.items() if rel > 0}

    precision, recall, f1 = precision_recall_f1_at_k(retrieved, relevant_set, k)
    ap = average_precision_at_k(retrieved, relevant_set, k)
    ndcg = ndcg_at_k(retrieved, rel_map, k)

    precision_list.append(precision)
    recall_list.append(recall)
    f1_list.append(f1)
    ap_list.append(ap)
    ndcg_list.append(ndcg)

metrics = {
    "lsa_dims": lsa_dims,
    f"MAP@{k}": float(np.mean(ap_list)) if ap_list else 0.0,
    f"NDCG@{k}": float(np.mean(ndcg_list)) if ndcg_list else 0.0,
    f"Precision@{k}": float(np.mean(precision_list)) if precision_list else 0.0,
    f"Recall@{k}": float(np.mean(recall_list)) if recall_list else 0.0,
    f"F1@{k}": float(np.mean(f1_list)) if f1_list else 0.0,
}

metrics

{'lsa_dims': 200,
 'MAP@10': 0.24075105911705108,
 'NDCG@10': 0.33795955139898953,
 'Precision@10': 0.23955555555555558,
 'Recall@10': 0.39951254861600577,
 'F1@10': 0.26999422560006786}